# MACSynDCR Models:

In [ ]:
num_epochs=200
learning_rate=0.0005

In [ ]:
import numpy as np
import pandas as pd
import csv
import glob
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

import keras
from keras import backend as K

from tensorflow.keras.layers import Input
from tensorflow.keras.models import Sequential,Model
from keras.layers import Dense, LSTM, Dropout, GRU, Bidirectional, Flatten, LSTM, Bidirectional
from keras.layers import Conv2D, MaxPooling2D, BatchNormalization
# from tensorflow.keras.layers import BatchNormalization
#from keras.layers.advanced_activations import LeakyReLU
from keras.layers import ELU, PReLU, LeakyReLU
from tensorflow.keras.optimizers import RMSprop,Adam, SGD
import tensorflow as tf
from sklearn.utils import class_weight
#from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split

import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from sklearn.metrics import average_precision_score
from scipy.stats import pearsonr
import tensorflow as tf
from tensorflow.keras import layers, models
import torch
import torch.nn as nn
from keras.models import load_model

import argparse
import os
import time
import pickle
import logging


import numpy as np
import pandas as pd
import csv
import glob
import tensorflow as tf
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,auc,roc_auc_score,precision_recall_curve,mean_squared_error
import numpy as np
import pandas as pd
import json
import random
from torchsummary import summary
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    auc,
    precision_recall_curve,
    mean_squared_error
)
loss_func = nn.MSELoss(reduction='sum')

import warnings
warnings.filterwarnings("ignore")


from datetime import datetime

import numpy as np
from sklearn.preprocessing import normalize
def to01(array):
    a = array.min()
    # ignore the Runtime Warning
    with np.errstate(divide='ignore'):
        b = 1. /(array.max() - array.min())
    if not(np.isfinite(b)):
        b = 0
    return np.vectorize(lambda x: b * (x - a))(array)



# Py Torch ANN
class MACSynDCR_ANN(nn.Module):
    def __init__(self):
        super(MACSynDCR_ANN, self).__init__()
        self.fc1 = nn.Linear(2432, 1024)
        self.bn1 = nn.BatchNorm1d(1024)
        #self.fc2 = nn.Linear(2048, 1024)
        #self.bn2 = nn.BatchNorm1d(1024)
        self.fc3 = nn.Linear(1024, 512)
        self.bn3 = nn.BatchNorm1d(512)
        self.fc4 = nn.Linear(512, 256)
        self.bn4 = nn.BatchNorm1d(256)
        self.fc5 = nn.Linear(256, 64)
        self.bn5 = nn.BatchNorm1d(64)
        self.fc6 = nn.Linear(64, 2)

    def forward(self, x):
        x = nn.ReLU()(self.bn1(self.fc1(x)))
        #x = nn.ReLU()(self.bn2(self.fc2(x)))
        x = nn.ReLU()(self.bn3(self.fc3(x)))
        x = nn.ReLU()(self.bn4(self.fc4(x)))
        x = nn.ReLU()(self.bn5(self.fc5(x)))
        x = F.softmax(self.fc6(x), dim=1)  # Softmax for output layer
        return x


class MACSynDCR_CNN(nn.Module):
    def __init__(self):
        super(MACSynDCR_CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(3, 3), padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), padding=1)
        self.fc1 = nn.Linear(32 * 68 * 32, 128)  # Adjusted for output shape of conv layers
        self.fc2 = nn.Linear(128, 1)  # Binary output
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)  # Add channel dimension
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x


#Keras ANN
def MACSynDCR_ANN_model2():
    model = keras.Sequential([
        layers.Input(shape=(2432,)),
        layers.Dense(1024, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
    print("MACSynDCR_ANN model:")
    model.summary()
    return model

def MACSynDCR_ANN_model():
    model = keras.Sequential([
        layers.Input(shape=(2432,)),
        layers.Dense(2048, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(1024, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dense(2, activation='softmax')
    ])
    #model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
    print("MACSynDCR_ANN model:")
    model.summary()
    return model

#Keras CNN
def MACSynDCR_CNN_model():
    model1 = Sequential()
    model1.add(Conv2D(32, (3, 3), input_shape = (64,38,1), activation='relu'))
    model1.add(MaxPooling2D(pool_size = (2, 2)))
    model1.add(Dropout(0.2))
    model1.add(Conv2D(64, (3, 3), activation='relu'))
    model1.add(MaxPooling2D(pool_size = (2, 2)))
    model1.add(Dropout(0.2))
    model1.add(Conv2D(128, (3, 3), activation='relu'))
    model1.add(MaxPooling2D(pool_size = (2, 2)))
    model1.add(Dropout(0.2))
    model1.add(Flatten())
    model1.add(Dense(128, activation = 'relu'))
    model1.add(Dense(2, activation = 'softmax'))

    print("MACSynDCR_CNN model:")
    model1.summary()
    return model1






def Ensemble_MACSynDCR_model2(ann_preds, cnn_preds,y_test):
    #ensemble_pred=np.array(preds).T
    print(ensemble_pred)
    summed = np.sum(ensemble_pred, axis=0)
    # argmax across classes
    mean_pred = np.mean(ensemble_pred, axis=1)
    #ensemble=[]
    j=0
    #for i in 1:
     #   ensemble.append(ensemble_pred[j,i])
     #   j=j+1
    ensemble=mean_pred
    print(ensemble)
    ensemble = (ensemble > 0.5).astype(int)
    ensemble_pred = (ensemble_pred > 0.5).astype(int)
    accuracy1 = accuracy_score(y_test, ensemble_pred[:,0])
    accuracy2 = accuracy_score(y_test, ensemble_pred[:,1])
    #accuracy3 = accuracy_score(y_test, ensemble_pred[:,2])
    #accuracy4 = accuracy_score(y_test, ensemble_pred[:,3])
    #accuracy5 = accuracy_score(y_test, ensemble_pred[:,4])
    ensemble_accuracy = accuracy_score(y_test, ensemble)

    print('Accuracy Score for Ensemble = ', accuracy1)
    print('Accuracy Score for LSTM = ', accuracy2)
    #print('Accuracy Score for LSTM = ', accuracy3)
    #print('Accuracy Score for LSTM = ', accuracy4)
    #print('Accuracy Score for GRU = ', accuracy5)
    print('Accuracy Score for ensemble = ', ensemble_accuracy)

    #Weighted average ensemble
    weights = [0.5, 0.3, 0.2]

    #Use tensordot to sum the products of all elements over specified axes.
    weighted_preds = np.tensordot(ensemble_pred, weights, axes=((0),(0)))
    weighted_ensemble_prediction = np.argmax(weighted_preds, axis=1)

    weighted_accuracy = accuracy_score(y_test, weighted_ensemble_prediction)

    print('Accuracy Score for model1 = ', accuracy1)
    print('Accuracy Score for model2 = ', accuracy2)
    #print('Accuracy Score for model3 = ', accuracy3)
    print('Accuracy Score for average ensemble = ', ensemble_accuracy)
    print('Accuracy Score for weighted average ensemble = ', weighted_accuracy)

    import pandas as pd
    df = pd.DataFrame([])

    for w1 in range(0, 5):
      #  for w2 in range(0,5):
            for w2 in range(0,5):
                wts = [w1/10.,w2/10.]
                wted_preds1 = np.tensordot(ensemble_pred, wts, axes=((0),(0)))
                wted_ensemble_pred = np.argmax(wted_preds1, axis=1)
                weighted_accuracy = accuracy_score(y_test, wted_ensemble_pred)
                df = pd.concat([df,pd.DataFrame({'wt1':wts[0],'wt2':wts[1],
                                             'acc':weighted_accuracy*100}, index=[0])], ignore_index=True)

                #df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    max_acc_row = df.iloc[df['acc'].idxmax()]
    print("Max accuracy of ", max_acc_row[0], " obained with w1=", max_acc_row[1],
          " w2=", max_acc_row[2])

    max(df['acc'])

    return ensemble

def Ensemble_MACSynDCR_model(train_pred,ann_preds,kann_preds, kcnn_preds,train_labels,y_test, fold):
    # Prepare features for stacking
    tann_preds=train_pred[0]
    tkann_preds=train_pred[1][:,1]
    tkcnn_preds=train_pred[2][:,1]

    ann_auc = roc_auc_score(y_test, ann_preds)
    ann_accuracy = accuracy_score(y_test, (ann_preds > 0.5).astype(int))
    print(f'Accuracy Score and AUC for ANN = AUC: {ann_auc:.4f}, Accuracy: {ann_accuracy:.4f}')

    kann_auc = roc_auc_score(y_test, kann_preds)
    kann_accuracy = accuracy_score(y_test, (kann_preds > 0.5).astype(int))
    print(f'Accuracy Score and AUC for KANN = AUC: {kann_auc:.4f}, Accuracy: {kann_accuracy:.4f}')

    kcnn_auc = roc_auc_score(y_test, kcnn_preds)
    kcnn_accuracy = accuracy_score(y_test, (kcnn_preds> 0.5).astype(int))
    print(f'Accuracy Score and AUC for KCNN = AUC: {kcnn_auc:.4f}, Accuracy: {kcnn_accuracy:.4f}')


    #stacked_X = np.column_stack((ann_preds,kann_preds, kcnn_preds))
    train_X = np.column_stack((tann_preds,tkann_preds, tkcnn_preds))
    test_X = np.column_stack((ann_preds,kann_preds,kann_preds))
    print(train_X.shape, type(test_X.shape))

    avg_pred=np.mean(test_X, axis=1)
    #mean_pred = np.mean(ensemble_pred, axis=1)
    max_pred=np.max(test_X, axis=1)
    avg_auc = roc_auc_score(y_test, avg_pred)
    avg_accuracy = accuracy_score(y_test, (avg_pred > 0.5).astype(int))
    print(f'Average AUC = AUC: {avg_auc:.4f}, Accuracy: {avg_accuracy:.4f}')

    max_auc = roc_auc_score(y_test, max_pred)
    max_accuracy = accuracy_score(y_test, (max_pred > 0.5).astype(int))
    print(f'MAX AUC = AUC: {max_auc:.4f}, Accuracy: {max_accuracy:.4f}')



    from sklearn.model_selection import GridSearchCV
    from sklearn.ensemble import RandomForestClassifier

    rf_model = RandomForestClassifier(random_state=42)
    param_grid = {
       'n_estimators': [50, 200],
       'max_depth': [None, 10, 30],
       'min_samples_split': [2, 10],
       'min_samples_leaf': [1, 4],
    }

    #grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid,
                              # scoring='accuracy', cv=3, verbose=0, n_jobs=-1)
    #grid_search.fit(train_X, train_labels)
   # best_rf_model = grid_search.best_estimator_
    #rf_ensemble_prob = best_rf_model.predict_proba(test_X)[:, 1]


    #{'C': [0.01, 0.1, 1, 10, 100]
    param_grid = {'C': [0.01,0.1, 1, 10,100], 'penalty': ['l1', 'l2']}
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5)
    grid_search.fit(train_X, train_labels)
    best_model = grid_search.best_estimator_
    #meta_model = LogisticRegression()
    #meta_model.fit(stacked_X, y_test)
    lr_ensemble_prob = best_model.predict_proba(test_X)[:,1]

    #rf_accuracy = accuracy_score(y_test, (rf_ensemble_prob> 0.5).astype(int))
    lr_accuracy = accuracy_score(y_test, (lr_ensemble_prob> 0.5).astype(int))
    if 0>=lr_accuracy:
        ensemble_prob=rf_ensemble_prob
        ensemble_pred=(ensemble_prob > 0.5).astype(int)
    else:
        ensemble_prob=lr_ensemble_prob
        ensemble_pred=(ensemble_prob > 0.5).astype(int)

    print(f"Finall ensemble results of fold_{fold+1} for MACSynDCR:")
    logging.info(f"Finall ensemble results of fold_{fold+1} for MACSynDCR:")
    logging.info("-" * 100)
    print("-" * 100)
    auc = roc_auc_score(y_test, ensemble_prob)
    accuracy = accuracy_score(y_test, ensemble_pred)
    auc_pr = average_precision_score(y_test,ensemble_prob)
    precision = precision_score(y_test, ensemble_pred)
    recall = recall_score(y_test, ensemble_pred)
    f1 = f1_score(y_test, ensemble_pred)
    yp=torch.tensor(ensemble_pred.astype(float))
    rmse = np.sqrt(loss_func(y_test.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    pcc, _ = pearsonr(y_test, ensemble_prob)

    print(f'Ensemble Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nEnsemble AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nEnsemble Recall: {recall:.4f}, F1 Score: {f1:.4f}')
    print(f'Ensemble RMSE: {rmse:.4f} , PCC: {pcc:.4f}')
    logging.info(f'Ensemble Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f}, \nEnsemble AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nEnsemble Recall: {recall:.4f}, F1 Score: {f1:.4f}, \nEnsemble RMSE: {rmse:.4f}')

    logging.info("-" * 100)
    print("-" * 100)
    return ensemble_pred, auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc


def Ensemble_MACSynDCR_model3(train_pred,ann_preds,kann_preds, kcnn_preds,train_labels,y_test, fold):
    # Prepare features for stacking
    tann_preds=train_pred[0]
    tkann_preds=train_pred[1][:,1]
    tkcnn_preds=train_pred[2][:,1]
    ann_auc = roc_auc_score(y_test, ann_preds)
    ann_accuracy = accuracy_score(y_test, (ann_preds.numpy() > 0.5).astype(int))
    print(f'Accuracy Score and AUC for ANN = AUC: {ann_auc:.4f}, Accuracy: {ann_accuracy:.4f}')

    kann_auc = roc_auc_score(y_test, kann_preds)
    kann_accuracy = accuracy_score(y_test, (kann_preds > 0.5).astype(int))
    print(f'Accuracy Score and AUC for KANN = AUC: {kann_auc:.4f}, Accuracy: {kann_accuracy:.4f}')

    kcnn_auc = roc_auc_score(y_test, kcnn_preds)
    kcnn_accuracy = accuracy_score(y_test, (kcnn_preds> 0.5).astype(int))
    print(f'Accuracy Score and AUC for KCNN = AUC: {kcnn_auc:.4f}, Accuracy: {kcnn_accuracy:.4f}')


    #stacked_X = np.column_stack((ann_preds,kann_preds, kcnn_preds))
    train_X = np.column_stack((tann_preds,tkann_preds, tkcnn_preds))
    test_X = np.column_stack((ann_preds,kann_preds, kcnn_preds))

    print(train_X.shape, test_X.shape)


    from sklearn.model_selection import GridSearchCV
    from sklearn.ensemble import RandomForestClassifier

    rf_model = RandomForestClassifier(random_state=42)
    param_grid = {
       'n_estimators': [50, 200],
       'max_depth': [None, 10, 30],
       'min_samples_split': [2, 10],
       'min_samples_leaf': [1, 4],
    }

    grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid,
                               scoring='accuracy', cv=3, verbose=0, n_jobs=-1)
    grid_search.fit(train_X, train_labels)
    best_rf_model = grid_search.best_estimator_
    rf_ensemble_prob = best_rf_model.predict_proba(test_X)[:, 1]


    #{'C': [0.01, 0.1, 1, 10, 100]
    param_grid = {'C': [0.01, 1, 100], 'penalty': ['l1', 'l2']}
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=3)
    grid_search.fit(train_X, train_labels)
    best_model = grid_search.best_estimator_
    #meta_model = LogisticRegression()
    #meta_model.fit(stacked_X, y_test)
    lr_ensemble_prob = best_model.predict_proba(test_X)[:, 1]

    rf_accuracy = accuracy_score(y_test, (rf_ensemble_prob> 0.5).astype(int))
    lr_accuracy = accuracy_score(y_test, (lr_ensemble_prob> 0.5).astype(int))
    if rf_accuracy>=lr_accuracy:
        ensemble_prob=rf_ensemble_prob
        ensemble_pred=(ensemble_prob > 0.5).astype(int)
    else:
        ensemble_prob=lr_ensemble_prob
        ensemble_pred=(ensemble_prob > 0.5).astype(int)

    print(f"Finall ensemble results of fold_{fold+1} for MACSynDCR:")
    logging.info(f"Finall ensemble results of fold_{fold+1} for MACSynDCR:")
    logging.info("-" * 100)
    print("-" * 100)
    auc = roc_auc_score(y_test, ensemble_prob)
    accuracy = accuracy_score(y_test, ensemble_pred)
    auc_pr = average_precision_score(y_test,ensemble_prob)
    precision = precision_score(y_test, ensemble_pred)
    recall = recall_score(y_test, ensemble_pred)
    f1 = f1_score(y_test, ensemble_pred)
    yp=torch.tensor(ensemble_pred.astype(float))
    rmse = np.sqrt(loss_func(y_test.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    pcc, _ = pearsonr(y_test, ensemble_prob)

    print(f'Ensemble Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nEnsemble AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nEnsemble Recall: {recall:.4f}, F1 Score: {f1:.4f}')
    print(f'Ensemble RMSE: {rmse:.4f} , PCC: {pcc:.4f}')
    logging.info(f'Ensemble Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f}, \nEnsemble AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nEnsemble Recall: {recall:.4f}, F1 Score: {f1:.4f}, \nEnsemble RMSE: {rmse:.4f}')

    logging.info("-" * 100)
    print("-" * 100)
    return ensemble_pred, auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc


# MACSynDCR Training:

In [ ]:
import numpy as np
import pandas as pd
import argparse
import os
import time
import pickle
import logging
import csv
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

import keras
from keras import backend as K

from tensorflow.keras.layers import Input
from tensorflow.keras.models import Sequential,Model
from keras.layers import Dense, LSTM, Dropout, GRU, Bidirectional, Flatten, LSTM, Bidirectional
from keras.layers import Conv2D, MaxPooling2D, BatchNormalization
# from tensorflow.keras.layers import BatchNormalization
#from keras.layers.advanced_activations import LeakyReLU
from keras.layers import ELU, PReLU, LeakyReLU
from tensorflow.keras.optimizers import RMSprop,Adam, SGD
from keras.models import load_model


import tensorflow as tf
from sklearn.utils import class_weight
#from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split

import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, mean_squared_error
from sklearn.metrics import average_precision_score
from scipy.stats import pearsonr
import tensorflow as tf
from tensorflow.keras import layers, models
import torch
import torch.nn as nn

from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    auc,
    precision_recall_curve,
    mean_squared_error
)


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset






import warnings
warnings.filterwarnings("ignore")


from datetime import datetime

def calc_stat(numbers):
    mu = sum(numbers) / len(numbers)
    sigma = (sum([(x - mu) ** 2 for x in numbers]) / len(numbers)) ** 0.5
    return mu, sigma

def save_args(args, save_to: str):
    args_dict = args.__dict__
    with open(save_to, 'w') as f:
        json.dump(args_dict, f, indent=2)


OUTPUT_DIR = 'output'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

n_delimiter = 100

import numpy as np
from sklearn.preprocessing import normalize
def to01(array):
    a = array.min()
    # ignore the Runtime Warning
    with np.errstate(divide='ignore'):
        b = 1. /(array.max() - array.min())
    if not(np.isfinite(b)):
        b = 0
    return np.vectorize(lambda x: b * (x - a))(array)



loss_func = nn.MSELoss(reduction='sum')

def Train_MACSynDCR_ANN(train_data, train_labels,val_data,val_labels, fold,ind_test):
        print("Data processing and Training of MACSynDCR_ANN model:")



        scaler = MinMaxScaler()
        train_data = scaler.fit_transform(train_data)
        val_data = scaler.fit_transform(val_data)
        ind_test = scaler.fit_transform(ind_test)

        train_data=torch.tensor(train_data).float()
        val_data=torch.tensor(val_data).float()
        ind_test=torch.tensor(ind_test).float()

        #label encoding y_train
        testy=val_labels
        label_encoder = preprocessing.LabelEncoder()
        y_train = label_encoder.fit_transform(train_labels)
        y_test = label_encoder.fit_transform(val_labels)



        y_test_cat = to_categorical(y_test,  num_classes = 2)
        y_train_cat = to_categorical(y_train, num_classes = 2)
        #print(y_test_cat.shape, y_train_cat.shape)

        model=MACSynDCR_ANN()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        criterion = nn.CrossEntropyLoss()

        #MSE loss
        loss_func = nn.MSELoss(reduction='sum')
        savefile="yes"
        model.train()
        best_val_acc=0.0
        for epoch in range(num_epochs):
            optimizer.zero_grad()
            outputs = model(train_data)#.detach().numpy()  # Ensure correct shape
            #print(outputs, outputs.shape)
            #v=np.argmax(outputs,axis=1).astype(float)
            #t=torch.FloatTensor(v)
            loss = criterion(outputs, train_labels.long())
            #trmse = np.sqrt(mean_squared_error(train_labels.float(), outputs.detach().numpy()))
            val_outputs = model(val_data).detach().numpy()
            val_predictions = np.argmax(val_outputs,axis=1)
            accuracy = accuracy_score(val_labels, val_predictions)
            AUC = roc_auc_score(val_labels, val_outputs[:,1])

            if accuracy > best_val_acc:
                best_preds=0.0
                best_val_acc = accuracy
                best_preds=val_outputs[:,1]
                #print(best_preds)
                train_preds=0.0
                train_pr= model(train_data).detach().numpy()
                train_preds=train_pr[:,1]

                ipreds=0.0
                i_pr= model(ind_test).detach().numpy()
                ipreds=i_pr[:,1]

                #best_model = model.state_dict()
                #torch.save(best_model, f'best_model_fold_{fold + 1}.pt')
                torch.save(model, f'/content/drive/MyDrive/MACSynDCR/saveModel/bestTANN_fold_{fold + 1}.h5')
                #torch.save(model, f'ANN_model_fold_{fold + 1}_ACC_{accuracy:.5f}_AUC_{AUC:.5f}.h5')
                #logging.info(f"ANN_model saved with Val_ACC: {accuracy:.4f}")
                print(f"ANN_model saved with Val_ACC: {accuracy:.4f} Val_AUC: {AUC:.4f}")
                logging.info("-" * n_delimiter)
                if savefile!="yes":
                    os.remove(savefile)
                #savefile=f'ANN_model_fold_{fold + 1}_ACC_{accuracy:.5f}_AUC_{AUC:.5f}.h5'

            loss.backward()
            optimizer.step()
            if (epoch%20)==0:
                print(f'fold_{fold+1}: Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, Val_ACC: {accuracy:.4f}, Val_AUC: {AUC:.4f}')
                logging.info(f'fold_{fold+1}: Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, ACC: {accuracy:.4f}, AUC: {AUC:.4f}')
        return model, best_preds,train_preds,ipreds






def Train_MACSynDCR_CNN(X,y,val_data,val_labels,fold):
    print("Data processing and Training of MACSynDCR_CNN model:")
    batch_size = 128
    #lr=0.001
    #num_epochs=100

    X_train_tensor = torch.tensor(X, dtype=torch.float32)
    y_train_tensor = torch.tensor(y, dtype=torch.float32)
    X_train_tensor=X_train_tensor.reshape(-1,64,38)

    val_data = torch.tensor(val_data, dtype=torch.float32)
    val_labels = torch.tensor(val_labels, dtype=torch.float32)
    val_data=val_data.reshape(-1,64,38)

   # print(X_train_tensor.shape)
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    model = MACSynDCR_CNN()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    best_val_acc=0.0
    savefile="yes"
    for epoch in range(num_epochs):
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.view(-1), labels.float())
            loss.backward()
            optimizer.step()
        #print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
        val_outputs = model(val_data).squeeze()
        val_predictions = (val_outputs > 0.5).float()
        accuracy = accuracy_score(val_labels.numpy(), val_predictions.numpy())
        AUC = roc_auc_score(val_labels, val_predictions)
        if accuracy > best_val_acc:
                best_val_acc = accuracy
                #best_model = model.state_dict()
                #torch.save(best_model, f'best_model_fold_{fold + 1}.pt')
                #torch.save(model, f'CNN_model_fold_{fold + 1}_{accuracy:.4f}.h5')
                torch.save(model, f'/content/drive/MyDrive/MACSynDCR/saveModel/CNN_model_fold_{fold + 1}.h5')
                #logging.info(f"CNN_model saved with Val_ACC: {accuracy:.4f}")
                print(f"CNN_model saved with Val_ACC: {accuracy:.4f}")
                logging.info("-" * n_delimiter)
                if savefile!="yes":
                    os.remove(savefile)
                #savefile=f'CNN_model_fold_{fold + 1}_{accuracy:.4f}.h5'
        if epoch%10==0:
            print(f'CNN fold_{fold+1}: Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, Val ACC: {accuracy:.4f}, , Val AUC: {AUC:.4f}')
            logging.info(f'CNN fold_{fold+1}: Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, Val ACC: {accuracy:.4f}, ,Val AUC: {AUC:.4f}')

    return model




#Keras models:

def Train_and_Validation_MACSynDCR_ANN(X_train,X_test,y_train,y_test,intTestData):
    print("Data processing and Training of MACSynDCR_ANN model:")
    #num_epochs =epoch
    batch_size = 128
   # learning_rate = 0.001
    #print(X_train.shape)
    #print(X_test.shape)

    scaler = MinMaxScaler()
    x_train = scaler.fit_transform(X_train)
    x_test = scaler.fit_transform(X_test)

    intTestData = scaler.fit_transform(intTestData)



    #label encoding y_train
    testy=y_test
    label_encoder = preprocessing.LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)
    y_test = label_encoder.fit_transform(y_test)

    y_test_cat = to_categorical(y_test,  num_classes = 2)
    y_train_cat = to_categorical(y_train, num_classes = 2)
    #print(y_test_cat.shape, y_train_cat.shape)


    model=MACSynDCR_ANN_model()
    model.save(f'/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_ANN_model.keras')
    model.compile(loss ='categorical_crossentropy', optimizer='adam',metrics =['acc','auc'])
    checkpoint_ann = ModelCheckpoint(
        '/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_ANN_model.keras',  # Filepath to save the model
        monitor='val_auc',  # Monitor validation accuracy
        save_best_only=True,  # Save only the best model
        mode='max',  # Maximize the monitored value
        verbose=1)

    os.remove(f'/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_ANN_model.keras')
    model.fit(x_train, y_train_cat, epochs=num_epochs, batch_size=batch_size,validation_data=(x_test, y_test_cat),callbacks=[checkpoint_ann])
    # Load the Keras model
    ANN = load_model('/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_ANN_model.keras')
    #model.save('MACSynDCR_ANN_model.hdf5')
    pred = ANN.predict(x_test)
    ann_pred=pred[:,1]
    ann_label=np.argmax(pred,axis=1)
    y_pred_prob=ann_pred
    y_pred=ann_label
    auc = roc_auc_score(y_test_cat, pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test_cat,pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    yp=torch.tensor(y_pred.astype(float))
    rmse = np.sqrt(loss_func(testy.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    pcc, _ = pearsonr(y_test_cat, pred)

    ipred = ANN.predict(intTestData)
    ann_ipred=ipred[:,1]

    print(f'ANN: Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nANN: AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nANN Recall: {recall:.4f}, F1 Score: {f1:.4f}')
    print(f'\ANN:RMSE: {rmse:.4f} , PCC: {pcc[1]:.4f}')
    return ann_ipred



def Train_and_Validation_MACSynDCR_Ensemble(X_train,X_test,y_train,y_val):
    print("Data processing and Training of MACSynDCR_CNN model:")
    #num_epochs = 200
    batch_size = 128
    #learning_rate = 0.001

    scaler = MinMaxScaler()
    x_train = scaler.fit_transform(X_train)
    x_test = scaler.fit_transform(X_test)

    #intTestData = scaler.fit_transform(intTestData)

    model=Model5()
    history = model.fit(x_train, y_train,
                    validation_data=(x_test, y_val),
                    epochs=num_epochs,
                    batch_size=128)

    X_train = x_train.reshape(-1, 64, 38)
    X_val = x_test.reshape(-1, 64, 38)
    model=MACSynDCR_Hybrid()
    model.compile(loss ='binary_crossentropy', optimizer='adam',metrics =['acc','auc'])
    checkpoint_ensemble = ModelCheckpoint(
        'best_MACSynDCR_Ensemble_model.keras',  # Filepath to save the model
        monitor='val_auc',  # Monitor validation accuracy
        save_best_only=True,  # Save only the best model
        mode='max',  # Maximize the monitored value
        verbose=1)
    history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=num_epochs,
                    batch_size=128,
                    callbacks=[checkpoint_ensemble])

    Ensemble = load_model('best_MACSynDCR_Ensemble_model.keras')
    # Evaluate the model on validation data
    loss, accuracy, AUC = Ensemble.evaluate(X_val, y_val)
    print(f'Evaluate->  Loss: {loss:.4f}, Accuracy: {accuracy:.4f},Accuracy: {AUC:.4f}')

    pred = Ensemble.predict(X_val).flatten()

    print(pred.shape,pred)
    #lstm_pred=pred[:,1]
    #ylabel=np.argmax(pred,axis=1)
    #y_pred_prob=cnn_pred
    #print(type(y_val),type(pred))
    y_test=y_val
    y_pred=(pred> 0.5).astype(int)
    auc = roc_auc_score(y_test, pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test,pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    #rmse = np.sqrt(mean_squared_error(y_test, pred))
    y_test=np.array(y_val).astype(float)
    #print(y_test.shape)
    #print(pred.shape)
    val_outputs = Ensemble.predict(X_val).squeeze()
    val_predictions = (val_outputs > 0.5).astype(float)
    rmse = np.sqrt(loss_func(y_test, val_predictions))

    #rmse = np.sqrt(mean_squared_error(train_labels.float(), outputs.detach().numpy()))

    pcc, _ = pearsonr(y_test, pred)
    #print(f'LSTM2: Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nLSTM: AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nLSTM2 Recall: {recall:.4f}, F1 Score: {f1:.4f},\nLSTM:RMSE: {rmse:.4f} , PCC: {pcc:.4f}')
    #print(f'\nLSTM:RMSE: {rmse:.4f} ')
    with np.printoptions(precision=4, suppress=True):
        print(f'Ensemble: Accuracy: {accuracy}, AUC-ROC: {auc} \nLSTM: AUC-PR: {auc_pr}, Precision: {precision}, \nLSTM2 Recall: {recall}, F1 Score: {f1},\nLSTM:RMSE: {rmse} , PCC: {pcc}')
    return pred, auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc[1]


# For keras


def Train_and_Validation_MACSynDCR_CNN(X_train,X_test,y_train,y_test,intTestData):
    print("Data processing and Training of MACSynDCR_CNN model:")
    #num_epochs = epoch
    batch_size = 128
    #learning_rate = 0.001

    scaler = MinMaxScaler()
    x_train = scaler.fit_transform(X_train)
    x_test = scaler.fit_transform(X_test)
    #print(x_train.shape)
    #print(x_test.shape)
    intTest = scaler.fit_transform(intTestData)

    #label encoding y_train
    testy=y_test
    label_encoder = preprocessing.LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)
    y_test = label_encoder.fit_transform(y_test)

    y_test_cat = to_categorical(y_test,  num_classes = 2)
    y_train_cat = to_categorical(y_train, num_classes = 2)
   # print(y_test_cat.shape, y_train_cat.shape)

    x_train = x_train.reshape(-1, 64, 38,1)
    x_test = x_test.reshape(-1, 64, 38,1)
    model=MACSynDCR_CNN_model()
    model.save(f'/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_CNN_model.keras')
    model.compile(loss ='categorical_crossentropy', optimizer='adam',metrics =['acc','auc'])
    checkpoint_cnn = ModelCheckpoint(
        '/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_CNN_model.keras',  # Filepath to save the model
        monitor='val_acc',  # Monitor validation accuracy
        save_best_only=True,  # Save only the best model
        mode='max',  # Maximize the monitored value
        verbose=1)

    os.remove(f'/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_CNN_model.keras')
    model.fit(x_train, y_train_cat, epochs=num_epochs, batch_size=batch_size,validation_data=(x_test, y_test_cat),callbacks=[checkpoint_cnn])
    # Load the Keras model
    CNN = load_model('/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_CNN_model.keras')
    #model.save('MACSynDCR_CNN_model.hdf5')
    pred = CNN.predict(x_test)
    #print(pred.shape,pred)
    cnn_pred=pred[:,1]
    cnn_label=np.argmax(pred,axis=1)
    y_pred_prob=cnn_pred
    y_pred=cnn_label
    auc = roc_auc_score(y_test_cat, pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test_cat,pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    yp=torch.tensor(y_pred.astype(float))
    rmse = np.sqrt(loss_func(testy.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    pcc, _ = pearsonr(y_test_cat, pred)

    intTest = intTest.reshape(-1, 64, 38,1)
    ipred = CNN.predict(intTest)
    #print(pred.shape,pred)
    cnn_ipred=ipred[:,1]

    print(f'CNN: Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nCNN: AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nCNN Recall: {recall:.4f}, F1 Score: {f1:.4f}')
    print(f'CNN:RMSE: {rmse:.4f} , PCC: {pcc[1]:.4f}')
    return cnn_ipred















#MACSynDCR Evaluation

In [ ]:
import argparse
import os
import time
import torch
import torch.nn as nn
import pickle
import logging


import numpy as np
import pandas as pd
import csv
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical


from sklearn.utils import class_weight
#from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential,Model
from keras.layers import Dense, LSTM, Dropout, GRU, Bidirectional, Flatten, LSTM, Bidirectional
import numpy as np
import pandas as pd
import argparse
import os
import time
import pickle
import logging
import csv
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

import keras
from keras import backend as K

from tensorflow.keras.layers import Input
from tensorflow.keras.models import Sequential,Model
from keras.layers import Dense, LSTM, Dropout, GRU, Bidirectional, Flatten, LSTM, Bidirectional
from keras.layers import Conv2D, MaxPooling2D, BatchNormalization
# from tensorflow.keras.layers import BatchNormalization
#from keras.layers.advanced_activations import LeakyReLU
from keras.layers import ELU, PReLU, LeakyReLU
from tensorflow.keras.optimizers import RMSprop,Adam, SGD
from keras.models import load_model


import tensorflow as tf
from sklearn.utils import class_weight
#from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    auc,
    precision_recall_curve,
    mean_squared_error
)
import numpy as np
import pandas as pd

from torch.utils.data import DataLoader, TensorDataset

import torch
import torch.nn as nn
import torch.optim as optim
from scipy.stats import pearsonr

import os
import pickle

import matplotlib.pyplot as plt
import torch
import json
import random



import warnings
warnings.filterwarnings("ignore")


from datetime import datetime

def calc_stat(numbers):
    mu = sum(numbers) / len(numbers)
    sigma = (sum([(x - mu) ** 2 for x in numbers]) / len(numbers)) ** 0.5
    return mu, sigma

def save_args(args, save_to: str):
    args_dict = args.__dict__
    with open(save_to, 'w') as f:
        json.dump(args_dict, f, indent=2)




OUTPUT_DIR = 'output'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

n_delimiter = 100


#batch_size = 256
#epochs = 500
#num_epochs=5
num_classes = 2



import numpy as np
from sklearn.preprocessing import normalize
def to01(array):
    a = array.min()
    # ignore the Runtime Warning
    with np.errstate(divide='ignore'):
        b = 1. /(array.max() - array.min())
    if not(np.isfinite(b)):
        b = 0
    return np.vectorize(lambda x: b * (x - a))(array)

loss_func = nn.MSELoss(reduction='sum')
def Eval_MACSynDCR_ANN(model, train_data,val_data,val_labels,fold):
    print("Evaluation of MACSynDCR_ANN:")
    logging.info("Evaluation of MACSynDCR_ANN model:")
    logging.info("-" * n_delimiter)
    print("-" * n_delimiter)
    vl=val_labels.float()
    model=torch.load(f'bestTANN_fold_{fold + 1}.h5')
    #model = torch.load(f'bestTANN_fold_{fold + 1}.h5')
    model.eval()
    with torch.no_grad():
            val_outputs = model(val_data).detach().numpy()
            val_predictions = np.argmax(val_outputs,axis=1)
            accuracy = accuracy_score(val_labels, val_predictions)
            AUC = roc_auc_score(val_labels, val_outputs[:,1])

            #val_outputs = model(val_data).squeeze()
            train_outputs = model(train_data).detach().numpy()[:,1]
            #val_predictions = (val_outputs > 0.5).float()
            precision = precision_score(val_labels, val_predictions)
            recall = recall_score(val_labels, val_predictions)
            f1 = f1_score(val_labels, val_predictions)
            #mlp_precision, mlp_recall, _ = precision_recall_curve(val_labels.numpy(), val_predictions.numpy())
            #AUC_PR = auc(mlp_precision, mlp_recall)
            AUC = roc_auc_score(val_labels, val_outputs[:,1])
            #print(val_labels.float(), val_predictions.float())
            #print(val_labels.shape,y_pred.shape)
            p=torch.tensor(val_predictions).float()
            #print(vl,p)
            vrmse = np.sqrt(loss_func(vl.float(),p.float() ))
            AUC_PR = average_precision_score(val_labels, val_outputs[:,1])
            pcc, _ = pearsonr(val_labels.float(), val_outputs[:,1])

            print(f'Accuracy: {accuracy:.4f}, AUC-ROC: {AUC:.4f} \n AUC-PR: {AUC_PR:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')
            print(f'\nRMSE: {vrmse:.4f} , PCC: {pcc:.4f}')
            logging.info(f'Accuracy: {accuracy:.4f}, AUC-ROC: {AUC:.4f}, \n AUC-PR: {AUC_PR:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}, RMSE: {vrmse:.4f}')

    logging.info("-" * n_delimiter)
    print("-" * n_delimiter)
    return val_outputs,train_outputs, AUC, accuracy, AUC_PR, f1, precision, recall, vrmse, pcc



def Eval_MACSynDCR_CNN(model,train_data,val_data,val_labels,fold):
    print("Evaluation of MACSynDCR_CNN:")
    logging.info("Evaluation of MACSynDCR_CNN model:")
    logging.info("-" * n_delimiter)
    print("-" * n_delimiter)
    vl=val_labels.float()
    val_data = torch.tensor(val_data, dtype=torch.float32)
    val_labels = torch.tensor(val_labels, dtype=torch.float32)
    val_data=val_data.reshape(-1,64,38)

    model = torch.load(f'CNN_model_fold_{fold + 1}.h5', weights_only=False)
    model.eval()
    with torch.no_grad():
        val_outputs = model(val_data).squeeze()
        val_predictions = (val_outputs > 0.5).astype(float)
        accuracy = accuracy_score(val_labels.numpy(), val_predictions.numpy())
        precision = precision_score(val_labels.numpy(), val_predictions.numpy())
        recall = recall_score(val_labels.numpy(), val_predictions.numpy())
        f1 = f1_score(val_labels.numpy(), val_predictions.numpy())
        AUC_PR = average_precision_score(val_labels, val_outputs)
        AUC = roc_auc_score(val_labels, val_predictions)
        rmse = np.sqrt(loss_func(vl.float(), val_predictions.float()))
        #rmse = np.sqrt(mean_squared_error(val_labels, val_predictions.float()))
        pcc, _ = pearsonr(val_labels, val_predictions.float())
        #vrmse = np.sqrt(loss_func(val_labels.float(), val_predictions.float()))

        print(f'Accuracy: {accuracy:.4f}, AUC-ROC: {AUC:.4f} \n AUC-PR: {AUC_PR:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')
        print(f'\nRMSE: {rmse:.4f} , PCC: {pcc:.4f}')

        logging.info(f'Accuracy: {accuracy:.4f}, AUC-ROC: {AUC:.4f} \n AUC-PR: {AUC_PR:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')
        logging.info(f'\nRMSE: {rmse:.4f} , PCC: {pcc:.4f}')

    logging.info("-" * n_delimiter)
    print("-" * n_delimiter)

    return val_outputs, AUC, accuracy, AUC_PR, f1, precision, recall, rmse, pcc






def Eval_MACSynDCR_ANN_model(train_data,val_data,val_labels,fold):
    print("Evaluation of Keras MACSynDCR_ANN model:")

    scaler = MinMaxScaler()
    val_data = scaler.fit_transform(val_data)
    vl=val_labels.float()
    testy=val_labels
    label_encoder = preprocessing.LabelEncoder()
    y_test = label_encoder.fit_transform(val_labels)

    y_test_cat = to_categorical(y_test,  num_classes = 2)

    ANN = load_model('/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_ANN_model.keras')
    loss, accuracy, AUC = ANN.evaluate(val_data, y_test_cat)
    print(f'Evaluate->  Loss: {loss:.4f}, Accuracy: {accuracy:.4f},AUC: {AUC:.4f}')
    #model.save('MACSynDCR_ANN_model.hdf5')
    #ANN.save(f'best_CV_MACSynDCR_ANN_model_Fold_{fold+1}_AUC_{AUC:.3f}_ACC_{accuracy:.3f}.keras')

    pred = ANN.predict(val_data)
    train_pred = ANN.predict(train_data)
    ann_pred=pred[:,1]
    ann_label=np.argmax(pred,axis=1)
    y_pred_prob=ann_pred
    y_pred=ann_label
    auc = roc_auc_score(y_test_cat, pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test_cat,pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    #rmse = np.sqrt(mean_squared_error(y_test, ann_pred))
    yp=torch.tensor(y_pred.astype(float))
    rmse = np.sqrt(loss_func(val_labels.float(), yp))
    pcc, _ = pearsonr(y_test_cat, pred)

    print(f'ANN: Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nANN: AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nANN Recall: {recall:.4f}, F1 Score: {f1:.4f}')
    print(f'ANN:RMSE: {rmse:.4f} , PCC: {pcc[1]:.4f}')
    print("*"*100)
    return ann_pred, train_pred



def Eval_MACSynDCR_CNN_model(train_data,val_data,val_labels, fold):
    print("Evaluation of Keras MACSynDCR_CNN model:")

    scaler = MinMaxScaler()
    val_data = scaler.fit_transform(val_data)

    testy=val_labels
    label_encoder = preprocessing.LabelEncoder()
    y_test = label_encoder.fit_transform(val_labels)

    y_test_cat = to_categorical(y_test,  num_classes = 2)

    val_data = val_data.reshape(-1, 64, 38,1)
    CNN = load_model('/content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_CNN_model.keras')
    loss, accuracy, AUC = CNN.evaluate(val_data, y_test_cat)
    print(f'Evaluate->  Loss: {loss:.4f}, Accuracy: {accuracy:.4f},Accuracy: {AUC:.4f}')
    #CNN.save(f'best_CV_MACSynDCR_CNN_model_Fold_{fold+1}_AUC_{AUC:.3f}_ACC_{accuracy:.3f}.keras')
    #model.save('MACSynDCR_ANN_model.hdf5')
    pred = CNN.predict(val_data)
    train_data = train_data.reshape(-1, 64, 38,1)
    train_pred = CNN.predict(train_data)
    cnn_pred=pred[:,1]
    cnn_label=np.argmax(pred,axis=1)
    y_pred_prob=cnn_pred
    y_pred=cnn_label
    auc = roc_auc_score(y_test_cat, pred)
    accuracy = accuracy_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test_cat,pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    yp=torch.tensor(y_pred.astype(float))
    rmse = np.sqrt(loss_func(testy.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, cnn_pred))
    pcc, _ = pearsonr(y_test_cat, pred)

    print(f'CNN: Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nCNN: AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nCNN Recall: {recall:.4f}, F1 Score: {f1:.4f}')
    print(f'CNN:RMSE: {rmse:.4f} , PCC: {pcc[1]:.4f}')
    print("*"*100)
    return cnn_pred, train_pred


# CV Train Main

In [ ]:
import argparse
import os
import time
import pickle
import logging


import numpy as np
import pandas as pd
import csv
import glob
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os
import pickle
import json
import random

from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    auc,
    precision_recall_curve,
    mean_squared_error
)


import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, KFold

#from models.MACSynDCR import DrugSynergyCNN,Model_ANN
#from models.MACSynDCR import MACSynDCR_CNN,MACSynDCR_LSTM
#from models.MACSynDCR_evaluation import Eval_MACSynDCR_CNN, Eval_MACSynDCR_LSTM



from tensorflow.keras.optimizers import RMSprop,Adam, SGD
from torchsummary import summary
#from models.MACSynDCR_model import Ensemble_MACSynDCR_model
from sklearn.linear_model import LogisticRegression

def Ensemble_MACSynDCR_model(out_dir,ann_preds,kann_preds, kcnn_preds,y_test, fold,val_index, indTest_pred,ind_y):

    ann_auc = roc_auc_score(y_test, ann_preds)
    ann_accuracy = accuracy_score(y_test, (ann_preds > 0.5).astype(int))
    print(f'Accuracy Score and AUC for ANN = AUC: {ann_auc:.4f}, Accuracy: {ann_accuracy:.4f}')

    kann_auc = roc_auc_score(y_test, kann_preds)
    kann_accuracy = accuracy_score(y_test, (kann_preds > 0.5).astype(int))
    print(f'Accuracy Score and AUC for KANN = AUC: {kann_auc:.4f}, Accuracy: {kann_accuracy:.4f}')

    kcnn_auc = roc_auc_score(y_test, kcnn_preds)
    kcnn_accuracy = accuracy_score(y_test, (kcnn_preds> 0.5).astype(int))
    print(f'Accuracy Score and AUC for KCNN = AUC: {kcnn_auc:.4f}, Accuracy: {kcnn_accuracy:.4f}')


    test_X = np.column_stack((ann_preds,kann_preds,kcnn_preds))
    #print(train_X.shape, type(test_X.shape))


    avg_pred=np.mean(test_X, axis=1)
    #mean_pred = np.mean(ensemble_pred, axis=1)
    df1 = pd.DataFrame({
    'sample_id': val_index,
    'probability': avg_pred
    })
    csv_file1 = os.path.join(out_dir, f'avg_prob{fold+1}.csv')
    df1.to_csv(csv_file1, index=False)

    max_pred=np.max(test_X, axis=1)
    df2 = pd.DataFrame({
    'sample_id': val_index,
    'probability': max_pred
    })

    csv_file2 = os.path.join(out_dir, f'max_prob{fold+1}.csv')
    df2.to_csv(csv_file2, index=False)

    avg_auc = roc_auc_score(y_test, avg_pred)
    avg_accuracy = accuracy_score(y_test, (avg_pred > 0.5).astype(int))
    print(f'Average AUC = AUC: {avg_auc:.4f}, Accuracy: {avg_accuracy:.4f}')

    max_auc = roc_auc_score(y_test, max_pred)
    max_accuracy = accuracy_score(y_test, (max_pred > 0.5).astype(int))
    print(f'MAX AUC = AUC: {max_auc:.4f}, Accuracy: {max_accuracy:.4f}')



    from sklearn.model_selection import GridSearchCV
    from sklearn.ensemble import RandomForestClassifier

    rf_model = RandomForestClassifier(random_state=42)
    param_grid = {
       'n_estimators': [50, 200],
       'max_depth': [None, 10, 30],
       'min_samples_split': [2, 10],
       'min_samples_leaf': [1, 4],
    }

    #grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid,
                              # scoring='accuracy', cv=3, verbose=0, n_jobs=-1)
    #grid_search.fit(train_X, train_labels)
   # best_rf_model = grid_search.best_estimator_
    #rf_ensemble_prob = best_rf_model.predict_proba(test_X)[:, 1]


    #{'C': [0.01, 0.1, 1, 10, 100]
    param_grid = {'C': [0.01,0.1, 1, 10,100,1000,10000], 'penalty': ['l1', 'l2']}
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5,verbose=1)
    grid_search.fit(test_X, y_test)
    best_model = grid_search.best_estimator_
    #meta_model = LogisticRegression()
    #meta_model.fit(stacked_X, y_test)
    lr_ensemble_prob = best_model.predict_proba(test_X)[:,1]
    ensemble_prob=lr_ensemble_prob
    ensemble_pred=(ensemble_prob > 0.5).astype(int)
    df3 = pd.DataFrame({
    'sample_id': val_index,
    'probability': ensemble_pred
    })
    csv_file3 = os.path.join(out_dir, f'ensemble_prob{fold+1}.csv')
    df3.to_csv(csv_file3, index=False)


    print(f"Finall ensemble results of fold_{fold+1} for MACSynDCR:")
    logging.info(f"Finall ensemble results of fold_{fold+1} for MACSynDCR:")
    logging.info("-" * 100)
    print("-" * 100)
    auc = roc_auc_score(y_test, ensemble_prob)
    accuracy = accuracy_score(y_test, ensemble_pred)
    auc_pr = average_precision_score(y_test,ensemble_prob)
    precision = precision_score(y_test, ensemble_pred)
    recall = recall_score(y_test, ensemble_pred)
    f1 = f1_score(y_test, ensemble_pred)
    yp=torch.tensor(ensemble_pred.astype(float))
    rmse = np.sqrt(loss_func(y_test.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    pcc, _ = pearsonr(y_test, ensemble_prob)

    print(f'Ensemble Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nEnsemble AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nEnsemble Recall: {recall:.4f}, F1 Score: {f1:.4f}')
    print(f'Ensemble RMSE: {rmse:.4f} , PCC: {pcc:.4f}')
    logging.info(f'Ensemble Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f}, \nEnsemble AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nEnsemble Recall: {recall:.4f}, F1 Score: {f1:.4f}, \nEnsemble RMSE: {rmse:.4f}')

    logging.info("-" * 100)
    print("-" * 100)

    ind_test_X = np.column_stack((indTest_pred[0],indTest_pred[1],indTest_pred[2]))
    grid_search2 = GridSearchCV(LogisticRegression(), param_grid, cv=5,verbose=1)
    grid_search2.fit(ind_test_X, ind_y)
    best_model2 = grid_search2.best_estimator_
    #meta_model = LogisticRegression()
    #meta_model.fit(stacked_X, y_test)
    ind_ensemble_prob = best_model2.predict_proba(ind_test_X)[:,1]
    ind_ensemble_pred=(ind_ensemble_prob > 0.5).astype(int)
    df4 = pd.DataFrame({
    'probability': ind_ensemble_prob,
    'prediction': ind_ensemble_pred,
    })
    csv_file4 = os.path.join(out_dir, f'ind_ensemble_prob{fold+1}.csv')
    df4.to_csv(csv_file4, index=False)
    ind_ensemble_results=[]
    auc = roc_auc_score(ind_y, ind_ensemble_prob)
    accuracy = accuracy_score(ind_y, ind_ensemble_pred)
    auc_pr = average_precision_score(ind_y,ind_ensemble_prob)
    precision = precision_score(ind_y, ind_ensemble_pred)
    recall = recall_score(ind_y, ind_ensemble_pred)
    f1 = f1_score(ind_y, ind_ensemble_pred)
    yp=torch.tensor(ind_ensemble_pred.astype(float))
    rmse = np.sqrt(loss_func(ind_y.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    pcc, _ = pearsonr(ind_y, ind_ensemble_prob)

    ind_ensemble_results.append(auc)
    ind_ensemble_results.append(accuracy)
    ind_ensemble_results.append(auc_pr)
    ind_ensemble_results.append(f1)
    ind_ensemble_results.append(precision)
    ind_ensemble_results.append(recall)
    ind_ensemble_results.append(rmse)
    ind_ensemble_results.append(pcc)
    ind_ensemble_results=np.array(ind_ensemble_prob)

    return ensemble_prob, auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc,ind_ensemble_results


import warnings
warnings.filterwarnings("ignore")


from datetime import datetime

def calc_stat(numbers):
    mu = sum(numbers) / len(numbers)
    sigma = (sum([(x - mu) ** 2 for x in numbers]) / len(numbers)) ** 0.5
    return mu, sigma

def save_args(args, save_to: str):
    args_dict = args.__dict__
    with open(save_to, 'w') as f:
        json.dump(args_dict, f, indent=2)




OUTPUT_DIR = '/content/drive/MyDrive/MACSynDCR/Data/output/CV_ind_testing/'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

n_delimiter = 100


#batch_size = 256
#epochs = 500
#num_epochs=5
num_classes = 2



import numpy as np
from sklearn.preprocessing import normalize
def to01(array):
    a = array.min()
    # ignore the Runtime Warning
    with np.errstate(divide='ignore'):
        b = 1. /(array.max() - array.min())
    if not(np.isfinite(b)):
        b = 0
    return np.vectorize(lambda x: b * (x - a))(array)



time_str = str(datetime.now().strftime('%y%m%d%H%M'))

#Training and Evaluation Function

input_size=2432
input_shape = (64, 38)
loss_func = nn.MSELoss(reduction='sum')
from sklearn.model_selection import StratifiedKFold

def lists_elements(lofl,i):
    el=[]
    for l in lofl:
        el.append(l[i])
    return el

def train_and_evaluate_model(data, labels, out_dir,intTestData,ind_y):


    #num_epochs=200
    #learning_rate=.0005

    #kfold=StratifiedKFold(n_splits=5, *, shuffle=True, random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    best_model = None
    best_accuracy = 0.0

    n_delimiter = 100


    fauc=[]
    facc=[]
    faucpr=[]
    ff1=[]
    fprec=[]
    frecall=[]
    frmse= []
    fpcc=[]
    ind_cv_results=[]

    pred_results = {
        'ann_pred': [],
        'cnn_pred': [],
        'lstm_pred': [],
        'Precision': [],
        'kann_pred': [],
        'kcnn_pred': [],
        'klstm_pred': [],
    }

    for fold, (train_index, val_index) in enumerate(kfold.split(data)):
        print(f'Fold {fold + 1}')
        logging.info(f'Fold {fold + 1}')
        logging.info("-" * n_delimiter)

        train_data, val_data = data[train_index], data[val_index]
        train_labels, val_labels = labels[train_index], labels[val_index]

        ANNmodel, best_preds, train_preds,ann_ipred=Train_MACSynDCR_ANN(train_data, train_labels,val_data,val_labels, fold,intTestData)
        print(ann_ipred)
        val_outputs = best_preds.astype(float)
        val_predictions = (val_outputs > 0.5).astype(float)
        #print(val_labels.numpy(),val_predictions,val_outputs)
        accuracy = accuracy_score(val_labels.numpy(), val_predictions)
        precision = precision_score(val_labels.numpy(), val_predictions)
        recall = recall_score(val_labels.numpy(), val_predictions)
        f1 = f1_score(val_labels.numpy(), val_predictions)
        auc_pr = average_precision_score(val_labels.float(), val_outputs.astype(float))
        auc = roc_auc_score(val_labels.float(), val_outputs.astype(float))
        rmse = np.sqrt(loss_func(val_labels.float(),torch.FloatTensor(val_outputs.squeeze())))
        #rmse = np.sqrt(mean_squared_error(val_labels, val_predictions.float()))
        #print(val_labels, torch.FloatTensor(val_outputs.squeeze()))
        pcc, _ = pearsonr(val_labels, torch.FloatTensor(val_outputs.squeeze()))
       # pcc, a = pearsonr([1,1,1], [1,1,4])
        #vrmse = np.sqrt(loss_func(val_labels.float(), val_predictions.float()))
        print(f'Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \n AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')
        print(f'RMSE: {rmse:.4f} , PCC: {pcc:.4f}')
        ann_pred=val_outputs
        ann_train_pred=train_preds

        kann_ipred=Train_and_Validation_MACSynDCR_ANN(train_data,val_data,train_labels,val_labels,intTestData)
        #print("KANN",kann_ipred)
        kcnn_ipred=Train_and_Validation_MACSynDCR_CNN(train_data,val_data,train_labels,val_labels,intTestData)
        #print("KCNN",kann_ipred)


        indTest_pred=[]
        #Eval_MACSynDCR_CNN(CNNmodel, train_data,val_data,val_labels,fold)
        #ann_pred, ann_train_pred,auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc=Eval_MACSynDCR_ANN(ANNmodel, train_data,val_data,val_labels,fold)
        kann_pred, kann_train_pred=Eval_MACSynDCR_ANN_model(train_data,val_data,val_labels,fold)
        kcnn_pred, kcnn_train_pred=Eval_MACSynDCR_CNN_model(train_data,val_data,val_labels,fold)
        indTest_pred.append(ann_ipred)
        indTest_pred.append(kann_ipred)
        indTest_pred.append(kcnn_ipred)

        ensemble_pred, auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc,ind_ensemble_results=Ensemble_MACSynDCR_model(out_dir,ann_pred,kann_pred, kcnn_pred,val_labels, fold,val_index,indTest_pred,ind_y)
        fauc.append(auc)
        facc.append(accuracy)
        faucpr.append(auc_pr)
        ff1.append(f1)
        fprec.append(precision)
        frecall.append(recall)
        frmse.append(rmse)
        fpcc.append(pcc)
        #ensemble_prob=np.array(ensemble_prob)
        ind_cv_results.append(ind_ensemble_results)


    #mu, sigma = calc_stat(test_losses)
    print("*"*n_delimiter)
    logging.info("*" * n_delimiter)
    print("*           Final MACSynCDR Results:")
    logging.info("*          Final MACSynCDR Results:")
    logging.info("*" * n_delimiter)
    print("*"*n_delimiter)
    mu, sigma = calc_stat(fauc)
    print(" AUC: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" AUC: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(facc)
    print(" Accuracy: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Accuracy: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(faucpr)
    print(" AUC-PR: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" AUC-PR: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(fprec)
    print(" Precision: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Precision: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(frecall)
    print(" Recall: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Recall: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(ff1)
    print(" F1: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" F1: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(frmse)
    print(" RMSE: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" RMSE: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(fpcc)
    print(" PCC: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" PCC: {:.4f} ± {:.4f}".format(mu, sigma))

    print("Max AUC: {:.4f}".format(max(fauc)))
    print("Max ACC: {:.4f}".format(max(facc)))
    print("Max AUC-PR: {:.4f}".format(max(faucpr)))
    print("Max Precision: {:.4f}".format(max(fprec)))
    print("Max Reacall: {:.4f}".format(max(frecall)))
    print("Max F1: {:.4f}".format(max(ff1)))
    print("MIN RMSE: {:.4f}".format(min(frmse)))
    print("Max PCC: {:.4f}".format(max(fpcc)))



    #### Test results



    print("*"*n_delimiter)
    logging.info("*" * n_delimiter)
    print("*           Independent MACSynCDR Testing Results:")
    logging.info("*          Independent MACSynCDR Testing Results:")
    logging.info("*" * n_delimiter)
    print("*"*n_delimiter)
    mu, sigma = calc_stat(lists_elements(ind_ensemble_results,0))
    print(" AUC: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" AUC: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_ensemble_results,1))
    print(" Accuracy: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Accuracy: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_ensemble_results,2))
    print(" AUC-PR: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" AUC-PR: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_ensemble_results,3))
    print(" Precision: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Precision: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_ensemble_results,4))
    print(" Recall: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Recall: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_ensemble_results,5))
    print(" F1: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" F1: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_ensemble_results,6))
    print(" RMSE: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" RMSE: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_ensemble_results,7))
    print(" PCC: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" PCC: {:.4f} ± {:.4f}".format(mu, sigma))




    return best_model


In [ ]:
def Ensemble_MACSynDCR_model(out_dir,ann_preds,kann_preds, kcnn_preds,y_test, fold,val_index, indTest_pred,ind_y):

    ann_auc = roc_auc_score(y_test, ann_preds)
    ann_accuracy = accuracy_score(y_test, (ann_preds > 0.5).astype(int))
    print(f'Accuracy Score and AUC for ANN = AUC: {ann_auc:.4f}, Accuracy: {ann_accuracy:.4f}')

    kann_auc = roc_auc_score(y_test, kann_preds)
    kann_accuracy = accuracy_score(y_test, (kann_preds > 0.5).astype(int))
    print(f'Accuracy Score and AUC for KANN = AUC: {kann_auc:.4f}, Accuracy: {kann_accuracy:.4f}')

    kcnn_auc = roc_auc_score(y_test, kcnn_preds)
    kcnn_accuracy = accuracy_score(y_test, (kcnn_preds> 0.5).astype(int))
    print(f'Accuracy Score and AUC for KCNN = AUC: {kcnn_auc:.4f}, Accuracy: {kcnn_accuracy:.4f}')


    val_X = np.column_stack((ann_preds,kann_preds,kcnn_preds))
    #print(train_X.shape, type(test_X.shape))


    avg_preds=np.mean(val_X, axis=1)
    #mean_pred = np.mean(ensemble_pred, axis=1)
    df1 = pd.DataFrame({
    'sample_id': val_index,
    'probability': avg_preds
    })
    csv_file1 = os.path.join(out_dir, f'avg_prob{fold+1}.csv')
    df1.to_csv(csv_file1, index=False)

    max_preds=np.max(val_X, axis=1)
    df2 = pd.DataFrame({
    'sample_id': val_index,
    'probability': max_preds
    })

    csv_file2 = os.path.join(out_dir, f'max_prob{fold+1}.csv')
    df2.to_csv(csv_file2, index=False)

    avg_auc = roc_auc_score(y_test, avg_preds)
    avg_accuracy = accuracy_score(y_test, (avg_preds > 0.5).astype(int))
    print(f'Average AUC = AUC: {avg_auc:.4f}, Accuracy: {avg_accuracy:.4f}')

    max_auc = roc_auc_score(y_test, max_preds)
    max_accuracy = accuracy_score(y_test, (max_preds > 0.5).astype(int))
    print(f'MAX AUC = AUC: {max_auc:.4f}, Accuracy: {max_accuracy:.4f}')



    from sklearn.model_selection import GridSearchCV
    from sklearn.ensemble import RandomForestClassifier

    rf_model = RandomForestClassifier(random_state=42)
    param_grid = {
       'n_estimators': [50, 200],
       'max_depth': [None, 10, 30],
       'min_samples_split': [2, 10],
       'min_samples_leaf': [1, 4],
    }

    #grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid,
                              # scoring='accuracy', cv=3, verbose=0, n_jobs=-1)
    #grid_search.fit(train_X, train_labels)
   # best_rf_model = grid_search.best_estimator_
    #rf_ensemble_prob = best_rf_model.predict_proba(test_X)[:, 1]


    #{'C': [0.01, 0.1, 1, 10, 100]
    param_grid = {'C': [0.01,0.1, 1, 10,100,1000,10000], 'penalty': ['l1', 'l2']}
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5,verbose=1)
    grid_search.fit(val_X, y_test)
    best_model = grid_search.best_estimator_
    #meta_model = LogisticRegression()
    #meta_model.fit(stacked_X, y_test)
    lr_ensemble_prob = best_model.predict_proba(val_X)[:,1]
    ensemble_prob=lr_ensemble_prob
    ensemble_pred=(ensemble_prob > 0.5).astype(int)
    df3 = pd.DataFrame({
    'sample_id': val_index,
    'probability': ensemble_pred
    })
    csv_file3 = os.path.join(out_dir, f'ensemble_prob{fold+1}.csv')
    df3.to_csv(csv_file3, index=False)


    print(f"Finall ensemble results of fold_{fold+1} for MACSynDCR:")
    logging.info(f"Finall ensemble results of fold_{fold+1} for MACSynDCR:")
    logging.info("-" * 100)
    print("-" * 100)
    auc = roc_auc_score(y_test, ensemble_prob)
    accuracy = accuracy_score(y_test, ensemble_pred)
    auc_pr = average_precision_score(y_test,ensemble_prob)
    precision = precision_score(y_test, ensemble_pred)
    recall = recall_score(y_test, ensemble_pred)
    f1 = f1_score(y_test, ensemble_pred)
    yp=torch.tensor(ensemble_pred.astype(float))
    rmse = np.sqrt(loss_func(y_test.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    pcc, _ = pearsonr(y_test, ensemble_prob)

    print(f'Ensemble Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \nEnsemble AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nEnsemble Recall: {recall:.4f}, F1 Score: {f1:.4f}')
    print(f'Ensemble RMSE: {rmse:.4f} , PCC: {pcc:.4f}')
    logging.info(f'Ensemble Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f}, \nEnsemble AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, \nEnsemble Recall: {recall:.4f}, F1 Score: {f1:.4f}, \nEnsemble RMSE: {rmse:.4f}')

    logging.info("-" * 100)
    print("-" * 100)

    ind_test_X = np.column_stack((indTest_pred[0],indTest_pred[1],indTest_pred[2]))
    grid_search2 = GridSearchCV(LogisticRegression(), param_grid, cv=5,verbose=1)
    grid_search2.fit(ind_test_X, ind_y)
    best_model2 = grid_search2.best_estimator_
    #meta_model = LogisticRegression()
    #meta_model.fit(stacked_X, y_test)
    ind_ensemble_prob = best_model2.predict_proba(ind_test_X)[:,1]
    ind_ensemble_pred=(ind_ensemble_prob > 0.5).astype(int)
    df4 = pd.DataFrame({
    'probability': ind_ensemble_prob,
    'prediction': ind_ensemble_pred,
    })
    csv_file4 = os.path.join(out_dir, f'ind_ensemble_prob{fold+1}.csv')
    df4.to_csv(csv_file4, index=False)
    ind_ensemble_results_metrics=[] # Renamed to avoid conflict
    iauc = roc_auc_score(ind_y, ind_ensemble_prob)
    iaccuracy = accuracy_score(ind_y, ind_ensemble_pred)
    iauc_pr = average_precision_score(ind_y,ind_ensemble_prob)
    iprecision = precision_score(ind_y, ind_ensemble_pred)
    irecall = recall_score(ind_y, ind_ensemble_pred)
    if1 = f1_score(ind_y, ind_ensemble_pred)
    yp=torch.tensor(ind_ensemble_pred.astype(float))
    irmse = np.sqrt(loss_func(ind_y.float(), yp))
    #rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    ipcc, _ = pearsonr(ind_y, ind_ensemble_prob)

    # Append the metrics to the list
    ind_ensemble_results_metrics.append(iauc)
    ind_ensemble_results_metrics.append(iaccuracy)
    ind_ensemble_results_metrics.append(iauc_pr)
    ind_ensemble_results_metrics.append(if1)
    ind_ensemble_results_metrics.append(iprecision)
    ind_ensemble_results_metrics.append(irecall)
    ind_ensemble_results_metrics.append(irmse)
    ind_ensemble_results_metrics.append(ipcc)

    # Return both the ensemble probabilities and the metrics
    return ensemble_prob, auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc, ind_ensemble_results_metrics

In [ ]:
def train_and_evaluate_model(data, labels, out_dir,intTestData,ind_y):
    #num_epochs=200
    #learning_rate=.0005

    #kfold=StratifiedKFold(n_splits=5, *, shuffle=True, random_state=42)
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    best_model = None
    best_accuracy = 0.0

    n_delimiter = 100


    fauc=[]
    facc=[]
    faucpr=[]
    ff1=[]
    fprec=[]
    frecall=[]
    frmse= []
    fpcc=[]
    ind_cv_results=[]

    pred_results = {
        'ann_pred': [],
        'cnn_pred': [],
        'lstm_pred': [],
        'Precision': [],
        'kann_pred': [],
        'kcnn_pred': [],
        'klstm_pred': [],
    }

    for fold, (train_index, val_index) in enumerate(kfold.split(data)):
        print(f'Fold {fold + 1}')
        logging.info(f'Fold {fold + 1}')
        logging.info("-" * n_delimiter)

        train_data, val_data = data[train_index], data[val_index]
        train_labels, val_labels = labels[train_index], labels[val_index]

        ANNmodel, best_preds, train_preds,ann_ipred=Train_MACSynDCR_ANN(train_data, train_labels,val_data,val_labels, fold,intTestData)
        print("TANN", ann_ipred)
        val_outputs = best_preds.astype(float)
        val_predictions = (val_outputs > 0.5).astype(float)
        #print(val_labels.numpy(),val_predictions,val_outputs)
        accuracy = accuracy_score(val_labels.numpy(), val_predictions)
        precision = precision_score(val_labels.numpy(), val_predictions)
        recall = recall_score(val_labels.numpy(), val_predictions)
        f1 = f1_score(val_labels.numpy(), val_predictions)
        auc_pr = average_precision_score(val_labels.float(), val_outputs.astype(float))
        auc = roc_auc_score(val_labels.float(), val_outputs.astype(float))
        rmse = np.sqrt(loss_func(val_labels.float(),torch.FloatTensor(val_outputs.squeeze())))
        #rmse = np.sqrt(mean_squared_error(val_labels, val_predictions.float()))
        #print(val_labels, torch.FloatTensor(val_outputs.squeeze()))
        pcc, _ = pearsonr(val_labels, torch.FloatTensor(val_outputs.squeeze()))
       # pcc, a = pearsonr([1,1,1], [1,1,4])
        #vrmse = np.sqrt(loss_func(val_labels.float(), val_predictions.float()))
        print(f'Accuracy: {accuracy:.4f}, AUC-ROC: {auc:.4f} \n AUC-PR: {auc_pr:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')
        print(f'RMSE: {rmse:.4f} , PCC: {pcc:.4f}')
        ann_pred=val_outputs
        ann_train_pred=train_preds

        kann_ipred=Train_and_Validation_MACSynDCR_ANN(train_data,val_data,train_labels,val_labels,intTestData)
        print("KANN",kann_ipred)
        kcnn_ipred=Train_and_Validation_MACSynDCR_CNN(train_data,val_data,train_labels,val_labels,intTestData)
        print("KCNN",kann_ipred)


        indTest_pred=[]
        #Eval_MACSynDCR_CNN(CNNmodel, train_data,val_data,val_labels,fold)
        #ann_pred, ann_train_pred,auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc=Eval_MACSynDCR_ANN(ANNmodel, train_data,val_data,val_labels,fold)
        kann_pred, kann_train_pred=Eval_MACSynDCR_ANN_model(train_data,val_data,val_labels,fold)
        kcnn_pred, kcnn_train_pred=Eval_MACSynDCR_CNN_model(train_data,val_data,val_labels,fold)
        indTest_pred.append(ann_ipred)
        indTest_pred.append(kann_ipred)
        indTest_pred.append(kcnn_ipred)

        # Capture the returned metrics list
        ensemble_prob, auc, accuracy, auc_pr, f1, precision, recall, rmse, pcc, ind_ensemble_results_metrics = Ensemble_MACSynDCR_model(out_dir,ann_pred,kann_pred, kcnn_pred,val_labels, fold,val_index,indTest_pred,ind_y)
        fauc.append(auc)
        facc.append(accuracy)
        faucpr.append(auc_pr)
        ff1.append(f1)
        fprec.append(precision)
        frecall.append(recall)
        frmse.append(rmse)
        fpcc.append(pcc)
        # Append the metrics list to ind_cv_results
        ind_cv_results.append(ind_ensemble_results_metrics)


    #mu, sigma = calc_stat(test_losses)
    print("*"*n_delimiter)
    logging.info("*" * n_delimiter)
    print("*           Final MACSynCDR Results:")
    logging.info("*          Final MACSynCDR Results:")
    logging.info("*" * n_delimiter)
    print("*"*n_delimiter)
    mu, sigma = calc_stat(fauc)
    print(" AUC: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" AUC: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(facc)
    print(" Accuracy: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Accuracy: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(faucpr)
    print(" AUC-PR: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" AUC-PR: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(fprec)
    print(" Precision: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Precision: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(frecall)
    print(" Recall: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Recall: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(ff1)
    print(" F1: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" F1: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(frmse)
    print(" RMSE: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" RMSE: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(fpcc)
    print(" PCC: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" PCC: {:.4f} ± {:.4f}".format(mu, sigma))

    print("Max AUC: {:.4f}".format(max(fauc)))
    print("Max ACC: {:.4f}".format(max(facc)))
    print("Max AUC-PR: {:.4f}".format(max(faucpr)))
    print("Max Precision: {:.4f}".format(max(fprec)))
    print("Max Reacall: {:.4f}".format(max(frecall)))
    print("Max F1: {:.4f}".format(max(ff1)))
    print("MIN RMSE: {:.4f}".format(min(frmse)))
    print("Max PCC: {:.4f}".format(max(fpcc)))



    #### Test results



    print("*"*n_delimiter)
    logging.info("*" * n_delimiter)
    print("*           Independent MACSynCDR Testing Results:")
    logging.info("*          Independent MACSynCDR Testing Results:")
    logging.info("*" * n_delimiter)
    print("*"*n_delimiter)
    mu, sigma = calc_stat(lists_elements(ind_cv_results,0))
    print(" AUC: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" AUC: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_cv_results,1))
    print(" Accuracy: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Accuracy: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_cv_results,2))
    print(" AUC-PR: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" AUC-PR: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_cv_results,3))
    print(" Precision: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Precision: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_cv_results,4))
    print(" Recall: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" Recall: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_cv_results,5))
    print(" F1: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" F1: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_cv_results,6))
    print(" RMSE: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" RMSE: {:.4f} ± {:.4f}".format(mu, sigma))

    mu, sigma = calc_stat(lists_elements(ind_cv_results,7))
    print(" PCC: {:.4f} ± {:.4f}".format(mu, sigma))
    logging.info(" PCC: {:.4f} ± {:.4f}".format(mu, sigma))

    return best_model

# Main function

In [ ]:
def main():
    out_dir = "/content/drive/MyDrive/MACSynDCR/Data/output/CV_ind_testing/"
    a=os.path.join(OUTPUT_DIR, 'cv_{}'.format(""))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    log_file = os.path.join(out_dir, 'cv.log')
    logging.basicConfig(filename=log_file,
                        format='%(asctime)s %(message)s',
                        datefmt='[%Y-%m-%d %H:%M:%S]',
                        level=logging.INFO)

    # Convert DataFrame to NumPy
    #dataset
    #drugdata = pd.read_csv('OneilProcessDataset30_dti.csv')
    #drugdata = pd.read_csv('OneilProcessDataset_0to10_2176_dti.csv')
    #drugdata = pd.read_csv('OneilProcessDataset_20to20_2176_dti.csv')
    #drugdata = pd.read_csv('OneilProcessDataset_all_2432_d1d2.csv')
    #drugdata = pd.read_csv('OneilProcessDataset_2176_d1cd2.csv')
    #drugdata = pd.read_csv('MFSynDCP_ProcessDataset_2176_d1cd2-10to10-Z.csv')
    drugdata = pd.read_csv('/content/drive/MyDrive/MACSynDCR/Data/DeepSyndergyProcessDataset_-10to20_2432_dti_LMGAN.csv')
    drugdata = pd.DataFrame(data=drugdata)
    xdata=drugdata.iloc[:,1:2433]
    ydata=drugdata.iloc[:,2433]
    Y = ydata
    X = xdata
    Y.value_counts()
    #xdata=to01(xdata.values)

    from sklearn import datasets
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import StratifiedKFold, cross_val_score

# -------- Step 1: 80-20 Split --------
    X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42)



    data = torch.FloatTensor(X_trainval.values)
    labels = torch.FloatTensor(y_trainval.values)

    X_test = torch.FloatTensor(X_test.values)
    y_test = torch.FloatTensor(y_test.values)

    best_model = train_and_evaluate_model(data, labels, out_dir,X_test,y_test) ##CV

if __name__ == "__main__":
    main()

Fold 1
Data processing and Training of MACSynDCR_ANN model:
ANN_model saved with Val_ACC: 0.5310 Val_AUC: 0.5130
fold_1: Epoch [1/200], Loss: 0.7040, Val_ACC: 0.5310, Val_AUC: 0.5130
ANN_model saved with Val_ACC: 0.7539 Val_AUC: 0.8708
ANN_model saved with Val_ACC: 0.8073 Val_AUC: 0.8944
ANN_model saved with Val_ACC: 0.8228 Val_AUC: 0.9056
ANN_model saved with Val_ACC: 0.8328 Val_AUC: 0.9146
ANN_model saved with Val_ACC: 0.8398 Val_AUC: 0.9215
ANN_model saved with Val_ACC: 0.8498 Val_AUC: 0.9280
ANN_model saved with Val_ACC: 0.8591 Val_AUC: 0.9333
ANN_model saved with Val_ACC: 0.8646 Val_AUC: 0.9382
ANN_model saved with Val_ACC: 0.8746 Val_AUC: 0.9423
ANN_model saved with Val_ACC: 0.8777 Val_AUC: 0.9454
ANN_model saved with Val_ACC: 0.8785 Val_AUC: 0.9473
ANN_model saved with Val_ACC: 0.8800 Val_AUC: 0.9499
ANN_model saved with Val_ACC: 0.8831 Val_AUC: 0.9513
ANN_model saved with Val_ACC: 0.8878 Val_AUC: 0.9521
ANN_model saved with Val_ACC: 0.8893 Val_AUC: 0.9556
fold_1: Epoch [21/200]

Model: "sequential_50"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_200 (Dense)               │ (None, 2048)           │     4,982,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_125         │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_201 (Dense)               │ (None, 1024)           │     2,098,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_126         │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_202 (Dense)               │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_127         │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_203 (Dense)               │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_128         │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_204 (Dense)               │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_129         │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_205 (Dense)               │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,786,114 (29.70 MB)

 Trainable params: 7,778,178 (29.67 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_87 (Conv2D)              │ (None, 62, 36, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_87 (MaxPooling2D) │ (None, 31, 18, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_87 (Dropout)            │ (None, 31, 18, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_88 (Conv2D)              │ (None, 29, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_88 (MaxPooling2D) │ (None, 14, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_88 (Dropout)            │ (None, 14, 8, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_89 (Conv2D)              │ (None, 12, 6, 128)     │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_89 (MaxPooling2D) │ (None, 6, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_89 (Dropout)            │ (None, 6, 3, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_29 (Flatten)            │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_238 (Dense)               │ (None, 128)            │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_239 (Dense)               │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 387,970 (1.48 MB)

 Trainable params: 387,970 (1.48 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 350ms/step - acc: 0.5283 - auc: 0.5384 - loss: 0.7379
Epoch 1: val_acc improved from -inf to 0.52090, saving model to /content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_CNN_model.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 19s 395ms/step - acc: 0.5281 - auc: 0.5382 - loss: 0.7371 - val_acc: 0.5209 - val_auc: 0.5224 - val_loss: 0.6922
Epoch 2/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 344ms/step - acc: 0.5102 - auc: 0.5081 - loss: 0.6937
Epoch 2: val_acc did not improve from 0.52090
41/41 ━━━━━━━━━━━━━━━━━━━━ 20s 378ms/step - acc: 0.5101 - auc: 0.5081 - loss: 0.6937 - val_acc: 0.5209 - val_auc: 0.5401 - val_loss: 0.6919
Epoch 3/200
41/41 ━━━━━━━━━━━━━━━━━━━━ 0s 341ms/step - acc: 0.5212 - auc: 0.5342 - loss: 0.6914
Epoch 3: val_acc improved from 0.52090 to 0.55805, saving model to /content/drive/MyDrive/MACSynDCR/saveModel/best_MACSynDCR_CNN_model.keras
41/41 ━━━━━━━━━━━━━━━━━━━━ 15s 376ms/step - acc: 0.5212 - auc: 0.5340 - loss: 0.6914 - val_acc: 0.5580 - va

# External validation using Arestozeneca dataset:

In [ ]:
import pandas as pd
drugdata = pd.read_csv('/content/drive/MyDrive/MACSynDCR/Data/AstraZeneca independent set.csv')
drugdata = pd.DataFrame(data=drugdata)
smilesdata = pd.read_csv('/content/drive/MyDrive/MACSynDCR/Data/Drug/smiles.csv')
smilesdata = pd.DataFrame(data=smilesdata)
drugdata

,drug1,drug2,cell,label
0,CNC(=O)CN1CCC(CC1)Oc2cc3c(cc2OC)ncnc3Nc4cccc(c...,c1cc(ccc1[C@H](CCO)NC(=O)C2(CCN(CC2)c3c4cc[nH]...,J82,1
1,CNC(=O)CN1CCC(CC1)Oc2cc3c(cc2OC)ncnc3Nc4cccc(c...,c1cc(ccc1[C@H](CCO)NC(=O)C2(CCN(CC2)c3c4cc[nH]...,SW780,0
2,C[C@@H]1CN(C[C@@H](N1)C)c2ccc(cc2)C(=O)Nc3cc(n...,c1cc(ccc1[C@H](CCO)NC(=O)C2(CCN(CC2)c3c4cc[nH]...,SW780,0
3,C[C@@H]1CN(C[C@@H](N1)C)c2ccc(cc2)C(=O)Nc3cc(n...,c1cc(ccc1[C@H](CCO)NC(=O)C2(CCN(CC2)c3c4cc[nH]...,TCCSUP,1
4,CS(=O)(=O)N1CCN(CC1)Cc2cc3c(s2)c(nc(n3)c4cccc5...,c1ccc(cc1)c2cc3c(ccn4c3n[nH]c4=O)nc2c5ccc(cc5)...,HCC1187,1
...,...,...,...,...
663,CN1CCN(CC1)CCOc2cc3c(c(c2)OC4CCOCC4)c(ncn3)Nc5...,c1ccc(cc1)c2cccc(c2N3CCC[C@H](C3)N)/C=C\4/C(=O...,TCCSUP,1
664,CN1CCC(CC1)COc2cc3c(cc2OC)c(ncn3)Nc4ccc(cc4F)Br,CS(=O)(=O)c1ccc(cc1)Nc2cncc(n2)c3cnc4n3cccc4,SW48,1
665,CN1CCC(CC1)COc2cc3c(cc2OC)c(ncn3)Nc4ccc(cc4F)Br,CS(=O)(=O)c1ccc(cc1)Nc2cncc(n2)c3cnc4n3cccc4,SW48,1
666,c1cn(c(=O)nc1N)C2C(C(C(O2)CO)O)(F)F,C[C@@H]1COCCN1c2cc(nc(n2)c3ccnc4c3cc[nH]4)C5(C...,SW780,1


In [ ]:
import pandas as pd

drugdataset = pd.read_csv('/content/drive/MyDrive/MACSynDCR/Data/AstraZeneca independent set.csv')
drugdataset2 = pd.read_csv("/content/drive/MyDrive/MACSynDCR/Data/OneilDataset_-10to20.csv")
drugsmile = smilesdata.SMILES

# Convert drugsmile list to a set for fast lookup
drugsmile_set = set(drugsmile)

# Extract unique drugs from 'cell' column in the second dataset
cell_set = set(drugdataset2['cell_line'].unique())

# Find intersection: drugs common to both drugsmile and 'cell' column
common_drugs = drugsmile_set & cell_set

# Filter the main drugdataset for instances where the drug is in the common list
# Assuming the relevant column in drugdataset is named 'drug' (update as needed)
filtered_drugdata = drugdataset[drugdataset['cell'].isin(drugdataset2.cell_line)]

# Show result
print(filtered_drugdata)
print(drugdataset['cell'].unique())
cell_set


Empty DataFrame
Columns: [drug1, drug2, cell, label]
Index: []
['J82' 'SW780' 'TCCSUP' 'HCC1187' 'HCC1806' 'HCC70' 'HCC1428' 'HCC1143'
 'HCC1395' 'HCC1937' 'HCC38' 'SW948' 'HCC1569' 'HCC1954' 'MCF7' 'SW900'
 'A549' 'HCC1419' 'RT4' 'SW48' 'C32' '22RV1' 'HCC1500' 'KATOIII']


{'A2058',
 'A2780',
 'A375',
 'A427',
 'CAOV3',
 'COLO320DM',
 'DLD1',
 'EFM192B',
 'ES2',
 'HCT116',
 'HT144',
 'HT29',
 'KPL1',
 'LNCAP',
 'LOVO',
 'MDAMB436',
 'MSTO',
 'NCIH1650',
 'NCIH2122',
 'NCIH23',
 'NCIH460',
 'NCIH520',
 'OCUBM',
 'OV90',
 'OVCAR3',
 'PA1',
 'RKO',
 'RPMI7951',
 'SKMEL30',
 'SKMES1',
 'SKOV3',
 'SW620',
 'SW837',
 'T47D',
 'UACC62',
 'UWB1289',
 'UWB1289BRCA1',
 'VCAP',
 'ZR751'}

In [ ]:
def main():
    out_dir = "/content/drive/MyDrive/MACSynDCR/Data/output/CV_External_validation/"
    a=os.path.join(OUTPUT_DIR, 'cv_{}'.format(""))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    log_file = os.path.join(out_dir, 'cv.log')
    logging.basicConfig(filename=log_file,
                        format='%(asctime)s %(message)s',
                        datefmt='[%Y-%m-%d %H:%M:%S]',
                        level=logging.INFO)


    #dataset

    #drugdata = pd.read_csv('MFSynDCP_ProcessDataset_2176_d1cd2-10to10-Z.csv')
    drugdata = pd.read_csv('/content/drive/MyDrive/MACSynDCR/Data/DeepSyndergyProcessDataset_-10to20_2432_dti_LMGAN.csv')
    drugdata = pd.DataFrame(data=drugdata)
    xdata=drugdata.iloc[:,1:2433]
    ydata=drugdata.iloc[:,2433]
    Y = ydata
    X = xdata
    Y.value_counts()
    #xdata=to01(xdata.values)

    from sklearn import datasets
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import StratifiedKFold, cross_val_score

# -------- Step 1: 80-20 Split --------
    #X_trainval, X_test, y_trainval, y_test = train_test_split(
   # X, Y, test_size=0.2, random_state=42)



    data = torch.FloatTensor(xdata.values)
    labels = torch.FloatTensor(ydata.values)

    X_test = torch.FloatTensor(X_test.values)
    y_test = torch.FloatTensor(y_test.values)

    best_model = train_and_evaluate_model(data, labels, out_dir,X_test,y_test) ##CV

if __name__ == "__main__":
    main()